In [ ]:
import os
import urllib.request
import zipfile
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# 1. 设定存放数据的本地目录
data_dir = './data'
os.makedirs(data_dir, exist_ok=True)

zip_path = os.path.join(data_dir, 'EuroSAT_RGB.zip')
extract_dir = data_dir  # 解压后的目录

# 2. 绕过 torchvision 崩溃的内置链接，直接使用 Zenodo 的官方永久稳定镜像
# zenodo_url = "https://zenodo.org/records/7711810/files/EuroSAT_RGB.zip"

# if not os.path.exists(extract_dir) or not os.listdir(extract_dir):
#     os.makedirs(extract_dir, exist_ok=True)
#     if not os.path.exists(zip_path):
#         print(f"正在从 Zenodo 下载数据集 (约90MB)，请稍候...")
#         try:
#             req = urllib.request.Request(zenodo_url, headers={'User-Agent': 'Mozilla/5.0'})
#             with urllib.request.urlopen(req) as response, open(zip_path, 'wb') as out_file:
#                 out_file.write(response.read())
#             print("下载完成！")
#         except Exception as e:
#             print(f"网络下载失败: {e}")
#             exit()

print("正在解压数据集...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)
print("解压完成！")

# 3. 智能寻找解压后的图像目录（绝对容错版）
dataset_root = extract_dir
for root, dirs, files in os.walk(extract_dir):
    # 只要该文件夹下包含 'Forest' 和 'River' 这两个经典的 EuroSAT 类别，就绝对是正确的根目录！
    if 'Forest' in dirs and 'River' in dirs:
        dataset_root = root
        break

print(f"🎯 已精准定位到数据集分类目录: {dataset_root}")

# 4. 数据预处理与加载：转为 32x32 以降低光计算硬件压力
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 使用 PyTorch 通用的 ImageFolder 加载
train_dataset = ImageFolder(root=dataset_root, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

print("="*50)
print(f"✅ 成功加载航空航天卫星数据集！")
print(f"🌍 图像总数: {len(train_dataset)} 张")
print(f"🛰️ 图像类别 (10分类): {train_dataset.classes}")
print("="*50)